# 📖 Error Propagation Examples

---

## 🎯 Learning Objectives

By the end of this notebook, you will:

1. **See** common error scenarios in action
2. **Understand** how errors propagate through agents
3. **Learn** to catch and handle these errors

---

## ⏱️ Time Estimate
**~30 minutes**

## 🧠 Why Errors Matter in Agents

The key difference from traditional software:
```
┌─────────────────────────────────────────────────────────────┐
│         TRADITIONAL vs AGENT ERRORS                       │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  TRADITIONAL:                                              │
│  1. Request → Code executes → Error thrown            │
│  2. Exception caught → Error message returned         │
│  3. Clear (what happened!)                            │
│                                                             │
│  AGENT:                                                   │
│  1. Prompt → LLM decides → Wrong tool selected       │
│  2. Tool executes → Wrong result                       │
│  3. LLM uses wrong result → Wrong final answer        │
│  4. SAYS nothing is wrong → Silent failure!          │
│                                                             │
│  ✗ Errors are harder to detect                        │
│  ✗ No clear error messages                            │
│  ✗ Can propagate silently                           │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
import os
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI Key: ")

from openai import OpenAI
client = OpenAI()

## 🔴 Error 1: Hallucination Cascade

In [ ]:
# Hallucination: Agent makes up a fake tool result

print("👻 ERROR 1: HALLUCINATION CASCADE")
print("=" * 60)
print("""
User: "What's the phone number for Apple's HQ?"

Step 1: Agent thinks: I don't know this, but I'll search.
Step 2: Agent "searches" (actually just guesses)
Step 3: Agent returns hallucinated result: "1-800-MY-IPHONE"

The error propagates: wrong search → hallucinated fact → wrong answer
""")

# Example in code
def hallucination_agent(user_query: str) -> str:
    """This agent hallucinates."""
    # The LLM doesn't know, so it makes up an answer
    if "phone" in user_query.lower() or "contact" in user_query.lower():
        # Hallucinated response 
        return "The phone number is 1-800-FAKE-NUMBER (this is made up!)"
    return "I don't have this information."

result = hallucination_agent("What's Apple's phone number?")
print(f"❌ Hallucinated result: {result}")
print("⚠️ EVAL CATCHES THIS: hallucination_evaluator would fail!")

## 🔴 Error 2: Tool Failure Propagation

In [ ]:
print("🔧 ERROR 2: TOOL FAILURE PROPAGATION")
print("=" * 60)
print("""
User: "What's the weather in Tokyo?"

Step 1: Agent decides to call get_weather tool
Step 2: Tool FAILS (API down, invalid location, etc.)
Step 3: Tool returns ERROR
Step 4: Agent doesn't handle error → returns broken response

The error propagates: tool error → unhandled → bad output
""")

# Example: Tool failure
def failing_tool(query: str) -> dict:
    """Simulates a failing tool."""
    raise Exception("API rate limit exceeded!")

def bad_agent_with_error(query: str) -> str:
    """Agent that doesn't handle tool errors."""
    try:
        result = failing_tool(query)
        return f"Weather: {result}"
    except Exception as e:
        # ❌ BAD: Just returns the error!
        return f"Error: {e}"

result = bad_agent_with_error("weather in Tokyo")
print(f"❌ Bad error handling: {result}")

# ✅ GOOD: Proper error handling
def good_agent_with_error(query: str) -> str:
    """Agent that handles errors properly."""
    try:
        result = failing_tool(query)
        return f"Weather: {result}"
    except Exception as e:
        # GOOD: Graceful degradation
        return "I couldn't get the weather right now. Let me try another approach."

result2 = good_agent_with_error("weather in Tokyo")
print(f"✅ Graceful handling: {result2}")

## 🔴 Error 3: Wrong Tool Selection

In [ ]:
print("🔧 ERROR 3: WRONG TOOL SELECTION")
print("=" * 60)
print("""
User: "What's 25 * 17?"

Step 1: Agent looks at tools: calculator, weather, search
Step 2: Agent ACCIDENTALLY uses weather tool
Step 3: Weather tool returns "Weather not available"
Step 4: Agent reports wrong info (weather instead of math)

The error propagates: wrong tool → wrong output → wrong final answer
""")

def wrong_tool_agent():
    """Simulates wrong tool selection."""
    query = "What is 25 * 17?"
    
    # Agent mistakes multiplication for weather somehow
    # (would happen with bad tool descriptions)
    
    # Wrong: Uses weather
    "weather_result" = "Weather data unavailable for expression"
    
    # Even worse: Agent makes this "correct"
    return f"The answer is {weather_result}"  # Wrong!

result = wrong_tool_agent()
print(f"❌ Wrong tool result: {result}")
print("⚠️ This needs: 1) Better tool descriptions 2) Eval catches it")

## 🔴 Error 4: State Corruption

In [ ]:
print("🔧 ERROR 4: STATE CORRUPTION")
print("=" * 60)
print("""
Conversation:
User 1: "My name is Alice"
User 2: "What's my name?"

Step 1: First turn → Agent stores "Alice" in memory
Step 2: Memory gets corrupted/overwritten
Step 3: Second turn → Agent doesn't remember "Alice"
Step 4: Agent says "I don't know" or wrong name

The error propagates: memory failure → lost context → wrong response
""")

# Simplified example
class BrokenMemory:
    """Memory with a bug."""
    def __init__(self):
        self.memory = ""
    
    def store(self, value: str):
        # BUG: Overwrites instead of appending!
        self.memory = value
    
    def recall(self) -> str:
        return self.memory

memory = BrokenMemory()
memory.store("Alice")  # Remember name
memory.store("Bob")    # Forgets Alice! Stores Bob only

print(f"❌ Corrupted memory: Should remember 'Alice' but got '{memory.recall()}'")

## 🟢 How Evals Catch These Errors

In [ ]:
print("✅ HOW EVALS CATCH ERRORS")
print("=" * 60)
print("""
1. HALLUCINATION EVAL:
   - Check: Does output align with known facts?
   - Method: Cross-reference with knowledge base
   - Fails when: Facts don't match

2. TOOL ERROR EVAL:
   - Check: Did tool calls succeed?
   - Method: Check tool response validity
   - Fails when: Tool error not detected

3. TOOL SELECTION EVAL:
   - Check: Right tool for the task?
   - Method: Expected tool matching
   - Fails when: Wrong tool called

4. MEMORY EVAL:
   - Check: Context preserved?
   - Method: Multi-turn test cases
   - Fails when: Context lost between turns

""")

## 💻 Writing Error-Catching Evals

In [ ]:
# These are the evals that WON'T catch these errors:
wrong_eval = lambda response: {"passed": True}  # Always passes!

# These are better:
def eval_hallucination(reference: str, prediction: str) -> dict:
    """Check if prediction matches known fact."""
    correct = reference.lower() in prediction.lower()
    return {
        "passed": correct,
        "feedback": "Hallucination detected" if not correct else "OK"
    }

def eval_tool_error(tool_result: dict) -> dict:
    """Check if tool executed successfully."""
    success = "error" not in tool_result and tool_result.get("status") != "error"
    return {
        "passed": success,
        "feedback": "Tool failed" if not success else "OK"
    }

def eval_memory(context: str, expected: str) -> dict:
    """Check if memory captured context."""
    captured = expected.lower() in context.lower()
    return {
        "passed": captured,
        "feedback": "Memory lost" if not captured else "OK"
    }

## ✅ Summary

Errors you now understand:
1. **Hallucination cascade** - Fake facts, wrong output
2. **Tool failures** - Errors not handled properly
3. **Wrong tool selection** - Wrong tool for task
4. **State corruption** - Lost context/memory

**Key**: Evals catch these errors! Build comprehensive evals.

## 🔗 Next
**[07_final_project_end_to_end.ipynb](07_final_project_end_to_end.ipynb)** - Build your complete agent!